In [34]:
import os
import json
import time
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd

import google.generativeai as genai

In [35]:


import os
import json
import time
from typing import List, Dict, Any, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
import google.generativeai as genai


API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY or API_KEY == "YOUR_API_KEY_HERE":
    raise ValueError("Please set GEMINI_API_KEY env var or replace YOUR_API_KEY_HERE with your actual key.")

genai.configure(api_key=API_KEY)

# Gemini model 
GEMINI_MODEL_NAME = "gemini-2.0-flash"



DEV_PATH = "../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl"
SAMPLE_PATH = "../Data/SemEval2026-Task_4-sample-v1/sample_track_a.jsonl"

print("Config ready.")
print("Using Gemini model:", GEMINI_MODEL_NAME)
print("Dev file path:", DEV_PATH)
print("Sample file path:", SAMPLE_PATH)


Config ready.
Using Gemini model: gemini-2.0-flash
Dev file path: ../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl
Sample file path: ../Data/SemEval2026-Task_4-sample-v1/sample_track_a.jsonl


In [47]:


df_dev = pd.read_json(DEV_PATH, lines=True)
print("Loaded dev set:", df_dev.shape)
print(df_dev.head(2))


Loaded dev set: (200, 4)
                                         anchor_text  \
0  The book follows an international organization...   
1  Glenn Tyler (Elvis Presley), a childish 25-yea...   

                                              text_a  \
0  The old grandmother Tina arrives in town to at...   
1  Bill Babbitt supported the death penalty, unti...   

                                              text_b  text_a_is_closer  
0  The nano-plague that poisoned Earth's water su...             False  
1  A white-collar suburban father Kyle (Fran Kran...              True  


In [48]:


event_model = genai.GenerativeModel(GEMINI_MODEL_NAME)
print("Gemini model initialized.")


Gemini model initialized.


In [38]:


EVENT_EXTRACTION_PROMPT = """
You MUST output ONLY valid JSON.
No explanations. No comments. No text outside the JSON object.

Extract the central event chain from the story.

Return EXACTLY ONE JSON OBJECT in this format:

{{
  "events": [
    {{
      "index": 1,
      "actors": ["actor1", "actor2"],
      "action": "main action verb phrase",
      "object": "object of action",
      "result": "what changed after the event",
      "is_outcome_event": true
    }}
  ]
}}

STORY:
{story}
"""


def extract_event_chain(
    story: str,
    max_retries: int = 3,
    sleep_sec: float = 0.5,
    print_raw_on_failure: bool = True
) -> Dict[str, Any]:
    """
    Use Gemini to extract an ordered list of events for a story.
    Returns: {"events": [ {index, actors, action, object, result, is_outcome_event}, ... ]}
    """
    last_raw = None

    for attempt in range(max_retries):
        try:
            prompt = EVENT_EXTRACTION_PROMPT.format(story=story)
            response = event_model.generate_content(prompt)
            text = response.text.strip()
            last_raw = text 

            # Strip code fences
            if text.startswith("```"):
                text = text.strip("`").strip()
                if text.lower().startswith("json"):
                    text = text[4:].strip()

            # First try direct parse
            try:
                data = json.loads(text)
            except json.JSONDecodeError:
                # Try to extract the first block
                first_brace = text.find("{")
                last_brace = text.rfind("}")
                if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
                    candidate = text[first_brace:last_brace+1]
                    data = json.loads(candidate)
                else:
                    raise

            # Validate structure
            events = data.get("events", [])
            if not isinstance(events, list):
                raise ValueError("'events' is not a list in parsed JSON.")

            # Normalize events
            normalized = []
            for idx, ev in enumerate(events):
                normalized.append({
                    "index": int(ev.get("index", idx + 1)),
                    "actors": [str(a).strip() for a in ev.get("actors", []) if str(a).strip()],
                    "action": str(ev.get("action", "")).strip(),
                    "object": str(ev.get("object", "")).strip(),
                    "result": str(ev.get("result", "")).strip(),
                    "is_outcome_event": bool(ev.get("is_outcome_event", False)),
                })

            return {"events": sorted(normalized, key=lambda e: e["index"])}

        except Exception as e:
            print(f"[extract_event_chain] Attempt {attempt+1} failed:", repr(e))
            time.sleep(sleep_sec)

    # After all attempts, optionally print raw text
    if print_raw_on_failure and last_raw is not None:
        print("\n--- RAW GEMINI OUTPUT (last attempt) ---")
        print(last_raw)
        print("--- END RAW OUTPUT ---\n")

    print("[extract_event_chain] Falling back to empty event chain.")
    return {"events": []}

print("Event extraction function ready.")


Event extraction function ready.


In [39]:

def debug_gemini_raw_output_for_row(df: pd.DataFrame, row_idx: int) -> None:
    story = df.loc[row_idx, "anchor_text"]
    prompt = EVENT_EXTRACTION_PROMPT.format(story=story)
    response = event_model.generate_content(prompt)
    print("------ RAW GEMINI OUTPUT ------")
    print(response.text)
    print("------ END ------")



In [40]:


def _tokenize(text: str) -> List[str]:
    if not text:
        return []
    tokens = []
    for tok in text.lower().replace(",", " ").replace(".", " ").split():
        tok = tok.strip()
        if tok:
            tokens.append(tok)
    return tokens

def jaccard_similarity(tokens_a: List[str], tokens_b: List[str]) -> float:
    set_a, set_b = set(tokens_a), set(tokens_b)
    if not set_a and not set_b:
        return 0.0
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union else 0.0

def event_similarity(ev1: Dict[str, Any], ev2: Dict[str, Any]) -> float:
    """
    Compute similarity between two events based on actors, action, object, and result.
    Returns a score in [0, 1].
    """

    actors1 = set(a.lower() for a in ev1.get("actors", []))
    actors2 = set(a.lower() for a in ev2.get("actors", []))
    actors_sim = jaccard_similarity(list(actors1), list(actors2))

    action_sim = jaccard_similarity(_tokenize(ev1.get("action", "")),
                                    _tokenize(ev2.get("action", "")))
    object_sim = jaccard_similarity(_tokenize(ev1.get("object", "")),
                                    _tokenize(ev2.get("object", "")))
    result_sim = jaccard_similarity(_tokenize(ev1.get("result", "")),
                                    _tokenize(ev2.get("result", "")))

    # Weighted combination
    return float(
        0.30 * actors_sim +
        0.30 * action_sim +
        0.20 * object_sim +
        0.20 * result_sim
    )

print("Event similarity helpers ready.")


Event similarity helpers ready.


In [41]:


def align_event_chains(chain_a: List[Dict[str, Any]],
                       chain_b: List[Dict[str, Any]]) -> Tuple[List[Tuple[int, int, float]], float]:
    """
    Greedy alignment: for each event in chain_a, find the best unmatched event in chain_b.
    Returns:
      - list of (idx_a, idx_b, sim)
      - average event similarity
    """
    if not chain_a or not chain_b:
        return [], 0.0

    aligned = []
    used_b = set()

    for i, ev_a in enumerate(chain_a):
        best_j = None
        best_sim = 0.0
        for j, ev_b in enumerate(chain_b):
            if j in used_b:
                continue
            sim = event_similarity(ev_a, ev_b)
            if sim > best_sim:
                best_sim = sim
                best_j = j
        if best_j is not None and best_sim > 0.0:
            used_b.add(best_j)
            aligned.append((i, best_j, best_sim))

    if not aligned:
        return [], 0.0

    avg_sim = float(np.mean([s for (_, _, s) in aligned]))
    return aligned, avg_sim

def sequence_order_score(aligned_pairs: List[Tuple[int, int, float]]) -> float:
    """
    Measures how well the order of events is preserved between chains.
    """
    if len(aligned_pairs) <= 1:
        return 1.0

    aligned_pairs = sorted(aligned_pairs, key=lambda t: t[0])
    b_indices = [j for (_, j, _) in aligned_pairs]

    total_pairs = 0
    correct_order = 0
    for i in range(len(b_indices)):
        for j in range(i + 1, len(b_indices)):
            total_pairs += 1
            if b_indices[i] < b_indices[j]:
                correct_order += 1

    if total_pairs == 0:
        return 1.0
    return correct_order / total_pairs

def outcome_similarity(chain_a: List[Dict[str, Any]],
                       chain_b: List[Dict[str, Any]]) -> float:
    """
    Compare outcome events between two chains.
    """

    def get_outs(chain: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        outs = [ev for ev in chain if ev.get("is_outcome_event", False)]
        if not outs and chain:
            outs = [chain[-1]]
        return outs

    outs_a = get_outs(chain_a)
    outs_b = get_outs(chain_b)

    if not outs_a or not outs_b:
        return 0.0

    scores = []
    for ev_a in outs_a:
        for ev_b in outs_b:
            scores.append(event_similarity(ev_a, ev_b))
    return float(max(scores)) if scores else 0.0

def event_chain_similarity(chain_a: List[Dict[str, Any]],
                           chain_b: List[Dict[str, Any]]) -> Dict[str, float]:
    """
    Full chain similarity:
      - avg_event_sim
      - order_score
      - outcome_sim
      - combined
    """
    aligned, avg_event_sim = align_event_chains(chain_a, chain_b)
    ord_score = sequence_order_score(aligned)
    out_sim = outcome_similarity(chain_a, chain_b)

    combined = float(
        0.45 * avg_event_sim +
        0.25 * ord_score +
        0.30 * out_sim
    )

    return {
        "avg_event_sim": avg_event_sim,
        "order_score": ord_score,
        "outcome_sim": out_sim,
        "combined": combined,
    }

print("Chain similarity functions ready.")


Chain similarity functions ready.


In [42]:

class NarrativeSimilarityResult:
    anchor_vs_a: Dict[str, float]
    anchor_vs_b: Dict[str, float]
    better: str     # "A" or "B"
    margin: float   # absolute difference between combined scores

def compare_anchor_candidates(
    anchor: str,
    text_a: str,
    text_b: str,
    cache: Dict[str, Dict[str, Any]] = None
) -> NarrativeSimilarityResult:
    """
    Compute event-chain similarity (anchor, A) and (anchor, B),
    and decide which is more similar.
    """
    if cache is None:
        cache = {}

    def get_chain(text: str) -> List[Dict[str, Any]]:
        if text in cache:
            return cache[text]["events"]
        chain = extract_event_chain(text)
        cache[text] = chain
        return chain["events"]

    anchor_chain = get_chain(anchor)
    a_chain = get_chain(text_a)
    b_chain = get_chain(text_b)

    anchor_vs_a = event_chain_similarity(anchor_chain, a_chain)
    anchor_vs_b = event_chain_similarity(anchor_chain, b_chain)

    score_a = anchor_vs_a["combined"]
    score_b = anchor_vs_b["combined"]

    better = "A" if score_a >= score_b else "B"
    margin = abs(score_a - score_b)

    return NarrativeSimilarityResult(
        anchor_vs_a=anchor_vs_a,
        anchor_vs_b=anchor_vs_b,
        better=better,
        margin=margin,
    )

print("Anchor A/B comparison function ready.")


Anchor A/B comparison function ready.


In [43]:
#single example from dev

row_idx = 0 
anchor_story = df_dev.loc[row_idx, "anchor_text"]
story_a = df_dev.loc[row_idx, "text_a"]
story_b = df_dev.loc[row_idx, "text_b"]
label_a_is_closer = bool(df_dev.loc[row_idx, "text_a_is_closer"])

print("Gold label: A is closer? ->", label_a_is_closer)

res = compare_anchor_candidates(anchor_story, story_a, story_b)

print("\nAnchor vs A:", res.anchor_vs_a)
print("Anchor vs B:", res.anchor_vs_b)
print("Predicted better match:", res.better, "(margin:", res.margin, ")")
print("Prediction A_is_closer? ->", res.better == "A")


Gold label: A is closer? -> False

Anchor vs A: {'avg_event_sim': 0.008695652173913044, 'order_score': 1.0, 'outcome_sim': 0.008695652173913044, 'combined': 0.2565217391304348}
Anchor vs B: {'avg_event_sim': 0.029052631578947368, 'order_score': 1.0, 'outcome_sim': 0.011764705882352941, 'combined': 0.2666030959752322}
Predicted better match: B (margin: 0.010081356844797384 )
Prediction A_is_closer? -> False


In [50]:
#Evaluate on a subset of dev data

def evaluate_on_dataframe(df: pd.DataFrame, max_rows: int = 50) -> float:
    """
    Evaluate event-chain model on first max_rows of df.
    """
    cache = {}
    preds = []
    gold = []

    n = min(max_rows, len(df))
    for idx in range(n):
        row = df.iloc[idx]
        anchor = row["anchor_text"]
        a = row["text_a"]
        b = row["text_b"]
        label_a_is_closer = bool(row["text_a_is_closer"])
        gold.append(label_a_is_closer)

        res = compare_anchor_candidates(anchor, a, b, cache=cache)
        pred_is_a = (res.better == "A")
        preds.append(pred_is_a)

        print(
            f"{idx+1}/{n}: pred={res.better}, gold_A={label_a_is_closer}, "
            f"margin={res.margin:.3f}, scores: A={res.anchor_vs_a['combined']:.3f}, B={res.anchor_vs_b['combined']:.3f}"
        )

    preds = np.array(preds, dtype=bool)
    gold = np.array(gold, dtype=bool)
    acc = float((preds == gold).mean())
    print(f"\nAccuracy on first {n} rows: {acc:.4f}")
    return acc

# Example usage
acc = evaluate_on_dataframe(df_dev, max_rows=5)


1/5: pred=B, gold_A=False, margin=0.246, scores: A=0.020, B=0.266
2/5: pred=B, gold_A=True, margin=0.021, scores: A=0.261, B=0.282
3/5: pred=B, gold_A=False, margin=0.016, scores: A=0.250, B=0.266
4/5: pred=A, gold_A=False, margin=0.244, scores: A=0.258, B=0.014
5/5: pred=B, gold_A=False, margin=0.001, scores: A=0.014, B=0.014

Accuracy on first 5 rows: 0.6000
